## Obtención de Actores desde TMDB y Generación de Scripts SQL

En este notebook obtenemos la información detallada de los actores de cada película del benchmark (3,830 películas) desde TMDB y generamos los scripts SQL para poblar la base de datos.

La base de datos del sistema tiene una estructura relacional many-to-many entre películas y actores:

- Tabla `actors`: actores únicos (id, nombre, foto de perfil)
- Tabla `movie_actors`: relación película-actor (movie_id, actor_id)

Extraemos los **5 actores principales** de cada película, ya que son los más representativos para mostrar en la interfaz de usuario.

In [2]:
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import requests
from dotenv import load_dotenv

from concurrent.futures import ThreadPoolExecutor, as_completed

# Load the environment variables from .env
load_dotenv()

True

### Funciones de Descarga

Reutilizamos la misma función genérica de los notebooks anteriores para consumir el endpoint `/credits` de la API de TMDB.

In [3]:
def fetch_tmdb_data(movie_id, api_key):
    """Generic function to fetch any TMDB data for a movie"""
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?api_key={api_key}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:  # Too Many Requests
            # Sleep for a bit if we hit rate limits
            time.sleep(2)
            return fetch_tmdb_data(movie_id, api_key)  # Simple retry
        else:
            print(f"Error: HTTP {response.status_code} for movie {movie_id}")
            return None
    except Exception as e:
        print(f"Error fetching data for movie {movie_id}: {e}")
        return None

In [4]:
def fetch_movies(ids, api_key, max_workers=10, delay=0.05):
    """
    Function to fetch movie data for multiple movie IDs concurrently
    
    Args:
        ids: List of movie IDs
        api_key: TMDB API key
        max_workers: Max number of concurrent requests
        delay: Small delay between requests to be nice to API
        
    Returns:
        (successful_results, failed_ids)
    """
    successful = []
    failed = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_tmdb_data, movie_id, api_key): movie_id for movie_id in ids}
        
        for future in as_completed(futures):
            movie_id = futures[future]
            result = future.result()
            
            if result:
                successful.append(result)
            else:
                failed.append(movie_id)
            
            # Small delay to avoid hammering the API
            time.sleep(delay)
    
    return successful, failed

### Carga de IDs del Catálogo

Cargamos los `tmdb_id` de las 3,830 películas del benchmark para descargar sus créditos.

In [5]:
ids = pd.read_csv('../data/benchmark/movies_benchmark.csv')[['tmdb_id']]
print(f'Películas en benchmark: {len(ids)}')
ids.head()

Películas en benchmark: 3830


,tmdb_id
0,862
1,8844
2,949
3,710
4,9087


In [6]:
api_key = os.getenv('tmdb_api_key')
movie_ids = ids['tmdb_id']

### Descarga de Créditos

In [7]:
# Fetching credits
credits, failed_credits = fetch_movies(movie_ids, api_key)
print(f"Fetched {len(credits)} movie credits, failed {len(failed_credits)}")

Fetched 3830 movie credits, failed 0


In [8]:
credits = pd.DataFrame(credits)
credits.head()

,id,cast,crew
0,10858,"[{'adult': False, 'gender': 2, 'id': 4173, 'kn...","[{'adult': False, 'gender': 2, 'id': 1152, 'kn..."
1,4584,"[{'adult': False, 'gender': 1, 'id': 7056, 'kn...","[{'adult': False, 'gender': 2, 'id': 1614, 'kn..."
2,710,"[{'adult': False, 'gender': 2, 'id': 517, 'kno...","[{'adult': False, 'gender': 2, 'id': 10702, 'k..."
3,949,"[{'adult': False, 'gender': 2, 'id': 1158, 'kn...","[{'adult': False, 'gender': 0, 'id': 15842, 'k..."
4,524,"[{'adult': False, 'gender': 2, 'id': 380, 'kno...","[{'adult': False, 'gender': 2, 'id': 1032, 'kn..."


### Preprocesamiento de Créditos

Del campo `cast` seleccionamos los **5 actores principales** (primeros en la lista de créditos) y extraemos solo los campos necesarios: `id` (TMDB), `name` y `profile_path`.

Ignoramos la información del equipo técnico (`crew`) ya que el director ya fue extraído en el Notebook 4.

In [9]:
# Process cast data: parse JSON, keep first 5 members with selected details
df = (
    credits[['id', 'cast']]
    .assign(
        cast=lambda d: d['cast'].apply(
            lambda x: [
                {"id": item["id"], "name": item["name"], "profile_path": item["profile_path"]}
                for item in (json.loads(x) if isinstance(x, str) else x)[:5]
            ]
        )
    )
    .rename(columns={'id': 'tmdb_id'})
)

df.head()

,tmdb_id,cast
0,10858,"[{'id': 4173, 'name': 'Anthony Hopkins', 'prof..."
1,4584,"[{'id': 7056, 'name': 'Emma Thompson', 'profil..."
2,710,"[{'id': 517, 'name': 'Pierce Brosnan', 'profil..."
3,949,"[{'id': 1158, 'name': 'Al Pacino', 'profile_pa..."
4,524,"[{'id': 380, 'name': 'Robert De Niro', 'profil..."


In [10]:
# Check the structure of the cast data
df.iloc[0]['cast']

[{'id': 4173,
  'name': 'Anthony Hopkins',
  'profile_path': '/dYVQTK1dPrQl1mugeLEWSSmA6Im.jpg'},
 {'id': 11148,
  'name': 'Joan Allen',
  'profile_path': '/aLYQFv11lb8hyBuP7ewkuxWbK8q.jpg'},
 {'id': 6280,
  'name': 'Powers Boothe',
  'profile_path': '/xj3DBMqnEnyYfHEBcRHOxlpbQS4.jpg'},
 {'id': 228,
  'name': 'Ed Harris',
  'profile_path': '/kUbUA70WPiosPT4kBJMWtGk0ASd.jpg'},
 {'id': 382,
  'name': 'Bob Hoskins',
  'profile_path': '/qOeE7bVHAsNTbP77Qk6TwrrWHZc.jpg'}]

Expandimos la lista de actores para que cada fila represente una única relación película-actor. El campo `cast` (lista de diccionarios) se convierte en columnas separadas (`id`, `name`, `profile_path`).

In [11]:
# Expand the cast list so each actor gets its own row
actors_df = df.explode('cast')

# Cast column contains dictionaries; split each dictionary into separate columns
actors_df = pd.concat(
    [
        actors_df.drop(columns=['cast']),
        actors_df['cast'].apply(lambda x: pd.Series(x) if isinstance(x, dict) else pd.Series())  # Extract actor fields
    ],
    axis=1
)

actors_df.head()

,tmdb_id,id,name,profile_path
0,10858,4173.0,Anthony Hopkins,/dYVQTK1dPrQl1mugeLEWSSmA6Im.jpg
0,10858,11148.0,Joan Allen,/aLYQFv11lb8hyBuP7ewkuxWbK8q.jpg
0,10858,6280.0,Powers Boothe,/xj3DBMqnEnyYfHEBcRHOxlpbQS4.jpg
0,10858,228.0,Ed Harris,/kUbUA70WPiosPT4kBJMWtGk0ASd.jpg
0,10858,382.0,Bob Hoskins,/qOeE7bVHAsNTbP77Qk6TwrrWHZc.jpg


### Verificación de Nulos

In [12]:
actors_df.isna().sum()

tmdb_id           0
id                6
name              6
profile_path    627
dtype: int64

Algunas películas de animación u otras sin reparto conocido pueden no tener información de actores. Eliminamos las filas donde falta el ID del actor para mantener la integridad referencial en la base de datos.

In [13]:
# Drop rows where actor ID is missing, and convert ID to integer
actors_df = actors_df.dropna(subset=['id'])
actors_df['id'] = actors_df['id'].astype(int)
actors_df.head()

,tmdb_id,id,name,profile_path
0,10858,4173,Anthony Hopkins,/dYVQTK1dPrQl1mugeLEWSSmA6Im.jpg
0,10858,11148,Joan Allen,/aLYQFv11lb8hyBuP7ewkuxWbK8q.jpg
0,10858,6280,Powers Boothe,/xj3DBMqnEnyYfHEBcRHOxlpbQS4.jpg
0,10858,228,Ed Harris,/kUbUA70WPiosPT4kBJMWtGk0ASd.jpg
0,10858,382,Bob Hoskins,/qOeE7bVHAsNTbP77Qk6TwrrWHZc.jpg


### Construcción de las Tablas Relacionales

Creamos dos DataFrames separados para las dos tablas de la base de datos:

1. **`actors_table`**: actores únicos (sin duplicados por `id` de TMDB)
2. **`movie_actor_table`**: relaciones película-actor (tmdb_id de la película + actor_id)

In [14]:
# Create a unique table of actors with their profile paths
actors_table = actors_df[['id', 'name', 'profile_path']].drop_duplicates(subset='id').reset_index(drop=True)
actors_table.head()

,id,name,profile_path
0,4173,Anthony Hopkins,/dYVQTK1dPrQl1mugeLEWSSmA6Im.jpg
1,11148,Joan Allen,/aLYQFv11lb8hyBuP7ewkuxWbK8q.jpg
2,6280,Powers Boothe,/xj3DBMqnEnyYfHEBcRHOxlpbQS4.jpg
3,228,Ed Harris,/kUbUA70WPiosPT4kBJMWtGk0ASd.jpg
4,382,Bob Hoskins,/qOeE7bVHAsNTbP77Qk6TwrrWHZc.jpg


In [15]:
# Keep only the TMDB ID and actor id
movie_actor_table = actors_df[['tmdb_id', 'id']].rename(columns={'id': 'actor_id'}).drop_duplicates()
movie_actor_table.head()

,tmdb_id,actor_id
0,10858,4173
0,10858,11148
0,10858,6280
0,10858,228
0,10858,382


In [16]:
# Since the database has an internal actor id, I will recreate the id for the actors and maintain consistency
actors_table['internal_id'] = range(1, len(actors_table) + 1)
actors_table.head()

,id,name,profile_path,internal_id
0,4173,Anthony Hopkins,/dYVQTK1dPrQl1mugeLEWSSmA6Im.jpg,1
1,11148,Joan Allen,/aLYQFv11lb8hyBuP7ewkuxWbK8q.jpg,2
2,6280,Powers Boothe,/xj3DBMqnEnyYfHEBcRHOxlpbQS4.jpg,3
3,228,Ed Harris,/kUbUA70WPiosPT4kBJMWtGk0ASd.jpg,4
4,382,Bob Hoskins,/qOeE7bVHAsNTbP77Qk6TwrrWHZc.jpg,5


In [17]:
# Merge movie_actor_table with actors_table to get the internal actor IDs
movie_actor_table = movie_actor_table.merge(
    actors_table[['id', 'internal_id']],
    left_on='actor_id',
    right_on='id',
    how='left'
).drop(columns=['id', 'actor_id']) \
 .rename(columns={'internal_id': 'actor_id'})

# Preview the result
movie_actor_table.head()

,tmdb_id,actor_id
0,10858,1
1,10858,2
2,10858,3
3,10858,4
4,10858,5


### Recuento de Actores y Relaciones

In [18]:
print(f"There are {actors_table.shape[0]} unique actors and {movie_actor_table.shape[0]} movie-actor relationships.")

There are 8960 unique actors and 18856 movie-actor relationships.


## Mapeo al ID Interno de Películas

La base de datos usa un `movie_id` interno (1-based) que es distinto al `movie_id` del CSV (0-based). Hacemos el merge con `movies_benchmark.csv` para obtener el `db_movie_id` correcto:

```
db_movie_id = movie_id (CSV, 0-based) + 1
```

Esto garantiza consistencia con los IDs de la tabla `movies` en la base de datos.

In [19]:
movies_df = pd.read_csv('../data/benchmark/movies_benchmark.csv')
print(f'Películas en benchmark: {len(movies_df)}')
movies_df[['movie_id', 'tmdb_id']].head()

Películas en benchmark: 3830


,movie_id,tmdb_id
0,0,862
1,1,8844
2,2,949
3,3,710
4,4,9087


In [20]:
# Keep only the internal movie_id and tmdb_id for mapping
movies_df = movies_df[['movie_id', 'tmdb_id']]
# Add one since our database IDs start from 1, but the CSV starts from 0
movies_df['db_movie_id'] = movies_df['movie_id'] + 1
movies_df.head()

,movie_id,tmdb_id,db_movie_id
0,0,862,1
1,1,8844,2
2,2,949,3
3,3,710,4
4,4,9087,5


In [21]:
# Merge to get the final movie-actor relationships with our database movie IDs
movie_actor = movies_df.merge(movie_actor_table, on='tmdb_id', how='inner')[['db_movie_id', 'actor_id']]
movie_actor = movie_actor.rename(columns={'db_movie_id': 'movie_id'})
movie_actor.head()

,movie_id,actor_id
0,1,29
1,1,30
2,1,24
3,1,31
4,1,32


### Generación de Scripts SQL

Generamos dos archivos SQL para insertar los datos en la base de datos:

- **`seed_actors.sql`**: inserta todos los actores únicos en la tabla `actors`
- **`seed_movie_actors.sql`**: inserta las relaciones película-actor en la tabla `movie_actors`

In [22]:
# ACTORS SQL FILE
with open('../../database/seed_actors.sql', 'w', encoding='utf-8') as f:
    f.write("INSERT INTO actors (tmdb_id, name, profile_path) VALUES\n")

    actor_values = []
    for _, row in actors_table.iterrows():
        tmdb_id = row['id']
        name = row['name'].replace("'", "''")  # Escape single quotes
        profile_path = row['profile_path'] if pd.notna(row['profile_path']) else 'NULL'
        profile_path = f"'{profile_path}'" if profile_path != 'NULL' else 'NULL'
        actor_values.append(f"({tmdb_id}, '{name}', {profile_path})")

    f.write(",\n".join(actor_values) + ";\n")

In [23]:
# MOVIE_ACTORS SQL FILE
with open('../../database/seed_movie_actors.sql', 'w', encoding='utf-8') as f:
    f.write("INSERT INTO movie_actors (movie_id, actor_id) VALUES\n")

    movie_actor_values = []
    for _, row in movie_actor.iterrows():
        movie_actor_values.append(f"({row['movie_id']}, {row['actor_id']})")

    f.write(",\n".join(movie_actor_values) + ";\n")

## Resumen del Notebook

En este notebook se obtuvieron los datos de actores para las 3,830 películas del benchmark y se generaron los scripts SQL para poblar la base de datos.

**Acciones realizadas:**
- Descarga de créditos (actores) para las 3,830 películas del benchmark desde TMDB
- Selección de los 5 actores principales por película
- Construcción de tabla de actores únicos con IDs internos
- Mapeo al `movie_id` de la base de datos (1-based)
- Generación de scripts SQL de inserción

**Outputs generados:**
- `database/seed_actors.sql` — actores únicos
- `database/seed_movie_actors.sql` — relaciones película-actor